[Guide Index](README.md)


# 03a SPARQL Entailment

In the rdflib-based `StarLayerGraph`, `query()` took an `entailment=` argument that applied an *entailment regime* to a single query, without first running `infer()` on the graph. **That query-time form is not yet available on `StarLayerGraph`**, the working name of what becomes `StarLayerGraph` (see the [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb)): `StarLayerGraph.query()` sends the SPARQL text straight to Oxigraph, and `query(..., entailment=...)` raises `NotImplementedError` until the entailment rewriting is ported. The `infer=` caching option that went with it is not available either.

What does work is the two-step form: `infer()` computes the closure, and you query the closure. Every entailment regime - what each one means, and a worked example of `infer()` - is explained once, in full, in the [Graph Inferencing guide](02b-graphs-inferencing.ipynb). This guide shows how to get the effect of query-time entailment with `infer()`, how to cache the entailed graph yourself, and how to detect an inconsistency via `ASK`.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier sections.


In [1]:
import pyoxigraph as ox
from starlayer.graph import ENTAILMENT
from starlayer.graph.oxgraph import StarLayerGraph

EX = "http://example.org/"
RDF_TYPE = ox.NamedNode("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

## 1. Applying entailment before the query

`g.infer(regime)` accepts the `ENTAILMENT.*` vocabulary (`ENTAILMENT.RDF`, `ENTAILMENT.RDFS`, `ENTAILMENT['OWL-RDF-Based']`, ...). By default (`mode="full"`) it returns a **new** `StarLayerGraph` holding the original triples plus the entailed ones; the graph you called it on is not changed. Query that closure instead of the original. (`mode="delta"` returns only the entailed triples and `mode="in-place"` adds them to the original.)

The RDF regime is computed natively on Oxigraph terms; RDFS and OWL RL run through `owlrl` on a temporary rdflib copy, and the result is converted back. A query that sets `entailment=` fails loudly rather than silently ignoring the regime (section 4).

In [2]:
g = StarLayerGraph(prefixes={"ex": EX})
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

QUERY = "SELECT ?x WHERE { ?x a ex:Employee }"

closure = g.infer(ENTAILMENT.RDFS)
print("plain (no entailment):     ", [x.x.value for x in g.query(QUERY)])
print("query on the RDFS closure: ", [x.x.value for x in closure.query(QUERY)])
print("original graph unchanged:", len(g), "triples; closure:", len(closure), "triples")

plain (no entailment):      []
query on the RDFS closure:  ['http://example.org/alice']
original graph unchanged: 2 triples; closure: 17 triples


## 2. Caching the entailed graph

`infer()` for `ENTAILMENT['OWL-RDF-Based']` is a potentially time-consuming calculation, and `StarLayerGraph` does not cache its result: every `infer()` call recomputes it. The `infer="changed"`, `"cached"` and `"always"` choices of the old query-time form are therefore not available. To get the same effect, keep the closure in a variable and decide yourself when to rebuild it: reuse it while the data has not changed (the old `"cached"`), call `infer()` again after an edit (the old `"always"` or `"changed"`).

In [3]:
# alice is a manager, but she is also a teamlead, because manager and teamlead are the same
g_owl = StarLayerGraph(prefixes={"ex": EX})
g_owl.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
""", format="turtle")

QUERY_OWL = "SELECT ?x WHERE { ?x a ex:TeamLead }"
OWL_RL = ENTAILMENT['OWL-RDF-Based']

def names(rows):
    return sorted(r.x.value.rsplit("/", 1)[1] for r in rows)

cached = g_owl.infer(OWL_RL)
print("before:", names(cached.query(QUERY_OWL)))

g_owl.add((ox.NamedNode(EX + "carol"), RDF_TYPE, ox.NamedNode(EX + "Manager")))
print("reusing the old closure (misses carol - stale on purpose):", names(cached.query(QUERY_OWL)))

cached = g_owl.infer(OWL_RL)   # rebuild after the change
print("after calling infer() again (picks up carol):             ", names(cached.query(QUERY_OWL)))

before: ['alice']
reusing the old closure (misses carol - stale on purpose): ['alice']
after calling infer() again (picks up carol):              ['alice', 'carol']


## 3. Detecting inconsistencies via `ASK`

`infer(ENTAILMENT['OWL-RDF-Based'])` inherits `owlrl`'s consistency checks (see the Inferencing guide's [section 4.5](02b-graphs-inferencing.ipynb#4.5-Detecting-inconsistencies) for what an inconsistency means and how it's flagged - an `err:error` triple, not an exception). `infer()` returns a closure even if it detects an inconsistency. Running a simple `ASK` query on the closure can detect the inconsistency:

In [4]:
ASK_INCONSISTENT = """
PREFIX err: <http://www.daml.org/2002/03/agents/agent-ont#>
ASK { ?e a err:ErrorMessage }
"""

# Dog and Cat are declared disjoint, but fido is asserted to be both
g_bad = StarLayerGraph(prefixes={"ex": EX})
g_bad.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
''', format="turtle")

bad_closure = g_bad.infer(OWL_RL)
print("g_bad (genuine inconsistency):  ", bad_closure.query(ASK_INCONSISTENT).askAnswer)
print("g (consistent):                 ", g_owl.infer(OWL_RL).query(ASK_INCONSISTENT).askAnswer)

# a normal query still returns a result - nothing is retracted, the inconsistency isn't surfaced here
rows = bad_closure.query("SELECT ?c WHERE { ex:fido a ?c }")
print("fido's classes (still returned despite the inconsistency):", sorted(r.c.value.rsplit("/", 1)[1] for r in rows))

g_bad (genuine inconsistency):   True
g (consistent):                  False
fido's classes (still returned despite the inconsistency): ['Cat', 'Dog']


## 4. The unbuilt query-time form

`query(..., entailment=...)` is the part of the old API that has no replacement yet. Rather than return a result that ignores the regime, it raises:

In [5]:
try:
    g.query(QUERY, entailment=ENTAILMENT.RDFS)
except NotImplementedError as error:
    print("NotImplementedError:", error)